# Capstone G1: Designing better CRISPR guides

Given a 20-letter guide RNA, will CRISPR-Cas9 cut efficiently there? You're building the tool that helps a scientist pick a guide *before* running a costly experiment.

**Your group's priority: interpretability.** A wet-lab scientist won't trust a black box telling them which guide to order -- your model has to show it's reading the sequence the way biology actually works.

This one is neither pixels nor a tidy table -- it's **text**: a DNA sequence. A computer can't
multiply the letter "A", so your first real decision is *how to turn a sequence into numbers*.
That choice matters more than the model. **Rubric:** build it, evaluate honestly, find a failure
mode, defend a decision, both partners can explain the work.

In [ ]:
# Setup: on Colab this grabs the course files + installs what's missing. Locally it's a no-op.
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g1_crispr")
sys.path.insert(0, "../..")            # day3_capstone (capstone_common / _tabular / _seq)
sys.path.insert(0, "../../../_shared") # colab_setup
import colab_setup
colab_setup.ensure(*colab_setup.DAY3_SEQ)

In [ ]:
import sys
sys.path.insert(0, "../..")
import capstone_seq as cs

df, meta = cs.load_guides()
print(meta["name"], "  ->", df.shape[0], "guides")
print(meta["citation"])
df[["guide30", "guide20", "gene", "gc_content", "efficiency", "high_efficiency"]].head()

## Reading a guide

Each row is a 30-letter window of DNA around one cut site:

```
[ 4 nt context ][   20 nt guide (protospacer)   ][ NGG PAM ][ 3 nt context ]
    1 - 4              5 - 24                       25 - 27      28 - 30
```

The **guide** (positions 5-24) is what you design; the **PAM** (NGG) is where Cas9 grabs on.
`efficiency` is how well that guide cut (0..1). `high_efficiency` = 1 for the best cutters.
The bases nearest the PAM -- the **seed region** -- are known to matter most. Keep that in mind.

## A first, human-readable guess: GC content
Before any ML, does one simple number -- the fraction of G/C bases -- track efficiency?

In [ ]:
cs.gc_vs_efficiency(df)

## Build your own model (interactive)

Two decisions: **how to featurize** the sequence, and **which model** to use.

- **representation** -- `onehot` keeps *where* each base sits (position-aware). `kmer` only
  counts how many of each base and base-pair (position-blind: it throws the order away).
- **sequence** -- `guide20` (just the guide) or `guide30` (guide + genomic context).
- **model** -- Logistic Regression, Random Forest, a small Neural Net, or CatBoost.

Because only ~1 in 5 guides is high-efficiency, **accuracy alone is misleading** -- a model that
says "low" every time scores 80%. Watch the **AUC** instead: 0.5 is guessing, 1.0 is perfect.

In [ ]:
from ipywidgets import interact_manual, Dropdown

def build(representation, sequence, model):
    cs.fit_eval_class(df, mode=representation, seq_col=sequence, model=model)

try:
    interact_manual(build,
        representation=Dropdown(options=["onehot", "kmer"], value="onehot", description="representation",
                                style={"description_width": "initial"}),
        sequence=Dropdown(options=["guide20", "guide30"], value="guide20", description="sequence"),
        model=Dropdown(options=list(cs.make_classifiers()), value="CatBoost", description="model"))
except ImportError:
    cs.fit_eval_class(df, mode="onehot", model="CatBoost")

### Compare onehot vs kmer

Run the builder twice -- once with `onehot`, once with `kmer`, same model. `onehot` should win.
Why? Because *where* a base sits matters (the seed region), and `kmer` throws that away. That gap
is a real result: **the representation, not the model, is doing the work.**

## Predict the exact efficiency (regression)

`high_efficiency` is a yes/no simplification. The real target is a *number* from 0 to 1. Can the
model predict it? R^2 = 0 means "no better than always guessing the average"; higher is better.

In [ ]:
from ipywidgets import interact_manual, Dropdown

def build_reg(representation, model):
    cs.fit_eval_reg(df, mode=representation, model=model)

try:
    interact_manual(build_reg,
        representation=Dropdown(options=["onehot", "kmer"], value="onehot", description="representation",
                                style={"description_width": "initial"}),
        model=Dropdown(options=list(cs.make_regressors()), value="Random Forest", description="model"))
except ImportError:
    cs.fit_eval_reg(df, mode="onehot", model="Random Forest")

## Interpretability: did the model learn real biology?

Your priority is **interpretability** -- not just *is it accurate*, but *is it right for the right
reason*. Train a linear model on the one-hot features and plot how much each **position** in the
guide matters. If the bases near the PAM (the right-hand end -- the seed) matter most, your model
rediscovered known CRISPR biology from data alone. That's your headline slide.

In [ ]:
cs.position_importance(df)

## Where to take it (with Claude)
- Filter to one **gene** and see if a model trained on the others still works on it (does guide design generalize across genes?).
- Engineer a feature by hand (e.g. 'is there a G right before the PAM?') and see if it helps.
- Look up a real guide-design tool (Benchling, CRISPOR) and compare its advice to your model's.